In [4]:
secfiles = [
    "sections-data-tool-shumei.json",
    "sections-nsfw.json",
    "sections-nsfw-shumei.json",
    "sections-safety240722.json",
]

import json

secdata = {}
seccounts = {}

for sf in secfiles:
    data = json.load(open(sf))
    secdata[sf] = data

    counts = seccounts[sf] = {}

    for k, v in data["count"].items():
        counts[k] = v

from pprint import pprint

pprint(seccounts)

{'sections-data-tool-shumei.json': {'ban': 4886,
                                    'blackandwhitelist': 21,
                                    'minor': 195,
                                    'normal': 118249,
                                    'politics': 24303,
                                    'porn': 4702,
                                    'sexy': 0,
                                    'star': 347,
                                    'violence': 923},
 'sections-data-tool2-shumei.json': {'normal': 181292,
                                     'politics': 82035,
                                     'porn': 903},
 'sections-nsfw-shumei.json': {'normal': 3111, 'porn': 4408},
 'sections-nsfw.json': {'drawings': 9646,
                        'hentai': 2597,
                        'neutral': 23629,
                        'porn': 355,
                        'sexy': 4176},
 'sections-safety240722.json': {'20240322_8k_nsfw_dup_1507': 1507,
                                '2024032

In [15]:
import random
from copy import deepcopy

collects = {
    "sections-data-tool-shumei.json": {
        "normal": {"train": 1.0, "val": 500, "label": "normal"},
        "politics": {"train": 6.0, "val": 1000, "label": "politics"},
        "porn": {"train": 8.0, "val": 500, "label": "porn"},
        # "violence": {"train": 10.0,"val": 50, "label": "violence"},
    },
    "sections-nsfw-shumei.json": {
        "normal": {"train": 8.0, "val": 500, "label": "normal"},
        "porn": {"train": 8.0, "val": 500, "label": "porn"},
    },
    # "sections-nsfw.json": {
    #     "drawings": {"train": 1.0, "label": "normal"},
    #     "neutral": {"train": 1.0, "label": "normal"},
    # },
    "sections-safety240722.json": {
        "20240322_8k_nsfw_dup_1507": {"train": 0.5, "val": 0, "label": "porn"},
        "20240329_163k_nsfw_dup_110k": {"train": 0.5, "val": 0, "label": "porn"},
        "nsfw_23k": {"train": 0.5, "val": 0, "label": "porn"},
        "sq_8k": {"train": 0.5, "val": 0, "label": "porn"},
    },
}

train_counts = {}
val_counts = {}
train_anns = []
val_anns = []
labels = []
label_map = {}

random.seed(0)

for sf, col in collects.items():
    # for each file, collect multiple sections
    counts = seccounts[sf]
    data = secdata[sf]

    for k, v in col.items():
        # collect sections

        label = v["label"]
        val_count = int(v["val"])
        train_count = int((counts[k] - val_count) * v["train"])

        # stat
        if label in train_counts:
            train_counts[label] += train_count
        else:
            train_counts[label] = train_count
        if label in val_counts:
            val_counts[label] += val_count
        else:
            val_counts[label] = val_count
        if label not in labels:
            labels.append(label)
            label_map[label] = len(labels) - 1
        label_id = label_map[label]

        # collect data
        files = deepcopy(data["files"][k])
        random.shuffle(files)

        # get val first to avoid overlap
        for _ in range(val_count):
            val_anns.append((files.pop(), label_id))

        # sample train
        files = files * int(v["train"] + 1)
        files = files[: int(train_count)]

        train_ann_ = [(f, label_id) for f in files]
        train_anns.extend(train_ann_)

pprint(train_counts)
pprint(val_counts)
pprint(train_anns[:10])
pprint(val_anns[:10])

assert set(train_anns) & set(val_anns) == set()

{'normal': 355304, 'politics': 210676, 'porn': 191407}
{'normal': 1000, 'politics': 1000, 'porn': 1000}
[('res2/陈光诚/google_陈光诚/00348.jpeg', 0),
 ('res2/西峯/google_西峯/00626.jpeg', 0),
 ('res2/袁莉/google_袁莉/00468.webp', 0),
 ('res2/割韭菜/google_割韭菜/00317.webp', 0),
 ('res/细颈瓶/google_细颈瓶/00346.jpeg', 0),
 ('res2/大奔女/google_大奔女/00149.png', 0),
 ('res2/梦鸽/google_梦鸽/00377.jpeg', 0),
 ('res/细颈瓶/google_细颈瓶/00379.jpeg', 0),
 ('res2/如胶似漆/google_如胶似漆/00362.jpeg', 0),
 ('res2/人民/google_人民/00139.jpeg', 0)]
[('res2/鸭绿江/google_鸭绿江/00491.jpeg', 0),
 ('res2/颜色革命/google_颜色革命/00076.jpeg', 0),
 ('res2/黄琦/google_黄琦/00331.jpeg', 0),
 ('res/平话/google_平话/00277.jpeg', 0),
 ('res/叼斤干/google_叼斤干/00010.jpeg', 0),
 ('res2/鸡蛋/google_鸡蛋/00500.jpeg', 0),
 ('res2/袁莉/google_袁莉/00211.jpeg', 0),
 ('res2/黄埔军校/google_黄埔军校/00465.jpeg', 0),
 ('res3/胡温新政/google_胡温新政/00551.jpeg', 0),
 ('res2/李婷婷/google_李婷婷/00062.jpeg', 0)]


In [7]:
v

{'train': 1.0, 'label': 'normal'}

In [7]:
with open('huangfan-shumei-0929-train.txt', 'w') as f:
    for ann in train_anns:
        f.write(f"{ann[0]} {ann[1]}\n")

with open('huangfan-shumei-0929-val.txt', 'w') as f:
    for ann in val_anns:
        f.write(f"{ann[0]} {ann[1]}\n")

with open('huangfan-shumei-0929-labels.txt', 'w') as f:
    for label in labels:
        f.write(f"{label}\n")